# Capstone — Content Refresh Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeremOzcn/flyrank-ml-internship-submission/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring

This notebook mirrors the deployed research paper at `site/index.html`. It runs the full pipeline: EDA → features → models → action engine. All results are reproducible with seed 42.

> Built on the [FlyRank ML Internship](https://flyrank.ai) dataset. All data is anonymized.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
# Research question: Can we predict which content pages are declining in search visibility

# and rank them by priority for a refresh action?print("Framing: Decision-support, not causal. We observe patterns, not predict Google's algorithm.")

#print("Decision: Priority-ranked refresh queue with action labels and reason codes.")

# Decision supported: A content manager receives a priority-ranked queue of pages,print("Question: Can we predict declining content pages and rank them for refresh?")

# each annotated with a suggested action and reason codes.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
# Data: content_refresh_anonymized.csv — 30,000 rows × 44 columns, 32 clients

# Teaching slice of the full 79M-row Hugging Face warehouse (FlyRank/internship-warehouse)print(f'Trend distribution: {df.trend_direction.value_counts().to_dict()}')

# Each row = one content page with 90-day aggregated GSC + GA4 metricsprint(f'Declining rate: {(df.trend_direction == "down").mean():.3f}')

#print(f'Clients: {df.client_id.nunique()}')

# Excluded as leakage: trend_direction, trend_pct (label-derived)print(f'Shape: {df.shape}')

# Excluded as identifiers: content_id, client_id (grouping only)df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')

# Excluded as non-features: provider_used, model_usedROOT = Path.cwd().parent.parent



import pandas as pdfrom pathlib import Path
import json

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
# Methodology:

# Label: is_declining = (trend_direction == 'down') — 54.2% positive rate    print(f"  {name:25s} ROC AUC: {m['roc_auc']:.3f} | AP: {m['average_precision']:.3f} | CV: {m['cv_roc_auc_mean']:.3f}±{m['cv_roc_auc_std']:.3f}")

# Features: 52 total (43 numeric + 9 categorical), engineered from raw columnsfor name, m in results['models'].items():

#   - Momentum: impression/click/session momentum (last30/prev30 ratio)print(f"Test clients: {results['n_test_clients']}")

#   - Ratios: CTR×impressions, engagement×sessions, scroll×CTR, position×impressionsprint(f"Split: {results['split']} (train={results['n_train']:,}, test={results['n_test']:,})")

#   - Flags: is_stale, is_very_stale, has_position, is_top10, measurable_opportunityprint(f"Best model: {results['best_model']}")

#   - Log transforms: log_impressions, log_clicks, log_sessions, etc.results = json.loads((ROOT / 'outputs' / 'capstone_model_results.json').read_text())

# Models: Logistic Regression, Random Forest, XGBoost, LightGBM, Extra Trees# Load pipeline results

# Validation: Client-holdout (80/20 by client) + 5-fold GroupKFold CV

# Baselines: Majority class, Random, Stale-first rule# Seed: 42

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# Results vs baseline

print("=== Model Results vs Baselines ===")    print(f"  {fi['feature']:30s} {fi['importance']:.4f}")

print(f"\nBest model: {results['best_model']}")for fi in results['feature_importance_top'][:5]:

best_metrics = results['models'][results['best_model']]print(f"\nFeature importance (top 5):")

baseline = results['baselines']['stale_first_rule']print(f"\n  Relative improvement over baseline: {(best_metrics['roc_auc'] - baseline['roc_auc']) / baseline['roc_auc'] * 100:.1f}%")

print(f"  ROC AUC:    {best_metrics['roc_auc']:.4f} (baseline: {baseline['roc_auc']:.4f})")print(f"  CV AUC:     {best_metrics['cv_roc_auc_mean']:.4f} ± {best_metrics['cv_roc_auc_std']:.4f}")

print(f"  Avg Precision: {best_metrics['average_precision']:.4f} (baseline: {baseline['average_precision']:.4f})")print(f"  Brier:      {best_metrics['brier_score']:.4f}")

print(f"  F1:         {best_metrics['f1']:.4f}")print(f"  P@50:       {best_metrics['precision_at_50']:.4f} (baseline: {baseline['precision_at_50']:.4f})")

## 5. Limitations

*What this work cannot claim.*

In [ ]:
# Limitations (honest framing):

# 1. Teaching slice (30K rows), not full 79M warehouse — results reflect the slice onlyprint("  5. 32 clients tested on 6 held-out — sufficient for capstone, not production")

# 2. Label is retrospective (pages that already declined), not predictive of future declineprint("  4. No causal claims — refresh effectiveness is a separate intervention question")

# 3. impression_momentum dominates (63% importance) — model detects decline, doesn't discover leading indicatorsprint("  3. Momentum dominates signal — model is a prioritization tool, not early warning")

# 4. No causal claims — refreshing a page may or may not reverse declineprint("  2. Retrospective label — detects existing decline, not future decline")

# 5. 32 clients, not 104 — cross-client generalization tested on 6 held-out clientsprint("  1. Teaching slice only (30K/79M rows) — designed to scale via DuckDB+hf://")

# 6. Rate columns are ×100 percentages (CTR=0.76 means 0.76%, not 76%)print("Limitations:")

# 7. avg_position=0 means no data, not position zero (1,205 rows)

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# Ranked recommendations (action playbook)

action_summary = json.loads((ROOT / 'outputs' / 'capstone_action_summary.json').read_text())print(f"Priority score p80: {action_summary['priority_score_stats']['p80']:.1f}")

print(f"Total pages scored: {action_summary['total_pages_scored']:,}")print(f"\nHigh-confidence pages: {action_summary['high_confidence_count']:,}")

print(f"\nAction distribution:")    print(f"  {reason:40s} {count:5d}")

for action, count in action_summary['action_distribution'].items():for reason, count in list(action_summary['top_reason_codes'].items())[:8]:

    print(f"  {action:35s} {count:5d}")print(f"\nTop reason codes:")

print(f"\nConfidence distribution:")    print(f"  {conf:10s} {count:5d}")
for conf, count in action_summary['confidence_distribution'].items():

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
# Artifacts the paper embeds:

# Charts in outputs/charts/:    print(f"  {f.name}")

#   - eda_label_distribution.png, eda_missingness.png, eda_per_client.pngfor f in sorted((ROOT / 'outputs').glob('capstone_*')):

#   - model_comparison.png, roc_curves.png, precision_at_k.pngprint(f"\nOutputs:")

#   - confusion_and_calibration.png, feature_importance_best.png    print(f"  {c.name}")

#   - per_client_performance.png, error_analysis.pngfor c in charts:

#   - action_engine_distribution.pngprint(f"Charts ({len(charts)}):")

# Data:charts = sorted((ROOT / 'outputs' / 'charts').glob('*.png'))

#   - outputs/capstone_model_results.jsonfrom pathlib import Path

#   - outputs/capstone_action_summary.json

#   - outputs/capstone_refresh_queue.csv (4,723 rows)#   - scripts/capstone_run_action_engine.py

#   - outputs/capstone_refresh_queue_top50.csv#   - scripts/capstone_run_model.py

#   - outputs/capstone_refresh_queue_top100.csv# Scripts:

# Notebooks:#   - work/notebooks/capstone_action_engine.ipynb

#   - work/notebooks/capstone_eda.ipynb#   - work/notebooks/capstone_model.ipynb
#   - work/notebooks/capstone_features.ipynb

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

---

## ML-12: Demo Outline + Social Post + Employer Summary

### 5-Minute Demo Outline

1. **Problem (30s):** "Content managers monitor thousands of pages. Which ones need a refresh? Today it's manual and reactive."
2. **Data (45s):** "30,000 pages across 32 anonymized clients. 90-day GSC + GA4 metrics. 54.2% are declining."
3. **Approach (60s):** "52 engineered features — momentum, ratios, flags. 5 models. Client-holdout validation — we test on clients the model has never seen."
4. **Results (60s):** "LightGBM: ROC AUC 0.995, Precision@50 = 1.000. The top 50 pages it flags are all genuinely declining. Baseline stale-first rule: 0.537."
5. **Action Engine (60s):** "Each page gets a priority score (0–100), an action label (refresh, review CTR, expand, protect, monitor), and reason codes. 1,400 high-confidence pages identified."
6. **Honest framing (45s):** "This is decision-support, not a crystal ball. The label is retrospective. Momentum dominates. But the ranking + reason codes save hours of manual triage."

### Social-Post Cut

> Built a content refresh ranking engine for 30K pages across 32 clients.
> LightGBM hits ROC AUC 0.995 — +85% over the stale-first baseline.
> Each page gets a priority score + action label + reason codes.
> Not a black box: the model says *why* a page is declining.
> Decision-support, not causal. Honest about limitations.
> 📄 Live paper: https://keremozcn.github.io/flyrank-ml-internship-submission/
> 📦 Code: https://github.com/KeremOzcn/flyrank-ml-internship-submission

### 3-Sentence Employer-Facing Summary

Built a decision-support ranking engine that scores 30,000 content pages by decline risk using
52 engineered features and five ML models (best: LightGBM, ROC AUC 0.995, +85% over baseline).
The action engine produces a daily priority queue with suggested actions (refresh, review CTR,
expand, protect, monitor) and human-readable reason codes — turning a reactive manual process
into a prioritized workflow. Validated with client-holdout splits and GroupKFold cross-validation
to ensure cross-client generalization.